# Kansas Property Tax Analysis - Data Extraction.
Extracting mill levy and valuation data from KDOR PDF reports (2020–2025) and Lincoln Institute 50-State Property Tax Comparison Study (2023–2025). 

In [16]:
import pdfplumber 
import pandas as pd 
import numpy as np 
import re 
import sqlalchemy 
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine 

## Parse Mill Levy PDFs (County Averages, 2020 - 2025)

In [23]:
columns = ['county', 'rural_2023', 'urban_2023', 'county_avg_2023', 'rural_2024', 'urban_2024', 'county_avg_2024', 'rural_2025', 'urban_2025', 'county_avg_2025']

def parse_mill_levy_pdf(filepath):
    all_rows = []
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            page_lines = page.extract_text().split(chr(10))
            for row_line in page_lines[5:]:
                parts = row_line.split()
                if len(parts) == 10:
                    all_rows.append(parts)
    return pd.DataFrame(all_rows, columns=columns)

df_2023_2025 = parse_mill_levy_pdf('../data/raw/kdor/23-25tableivavglevies.pdf')
df_2023_2025.head()




,county,rural_2023,urban_2023,county_avg_2023,rural_2024,urban_2024,county_avg_2024,rural_2025,urban_2025,county_avg_2025
0,Allen,144.705,204.769,164.063,142.848,204.141,162.286,141.876,201.992,160.752
1,Anderson,131.340,158.581,138.823,135.059,157.865,141.656,133.116,155.268,139.498
2,Atchison,113.469,152.251,131.723,116.212,153.119,134.394,121.105,158.678,140.048
3,Barber,153.542,197.229,161.383,168.618,200.857,175.607,139.631,184.289,147.720
4,Barton,141.697,166.732,154.307,139.299,162.306,150.971,137.62,157.025,147.689


## Confirmation that number of rows match number of Kansas Counties 

In [24]:
print(len(df_2023_2025))

105


## Parse Remaining Mill Levy PDFs

In [29]:
def parse_mill_levy_pdf_generic(filepath, years):
    col_names = ['county']
    for year in years:
        col_names.append(f'rural_{year}')
        col_names.append(f'urban_{year}')
        col_names.append(f'county_avg_{year}')
        all_rows = []
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                page_lines = page.extract_text().split(chr(10))
                for row_line in page_lines[5:]:
                    parts = row_line.split()
                    if len(parts) == 10:
                        all_rows.append(parts)
    return pd.DataFrame(all_rows, columns=col_names)

In [30]:
df_2022_2024 = parse_mill_levy_pdf_generic('../data/raw/kdor/22-24tableivavglevies.pdf', [2022, 2023, 2024]) 
df_2021_2023 = parse_mill_levy_pdf_generic('../data/raw/kdor/21-23tableivavglevies.pdf', [2021, 2022, 2023]) 
df_2020_2022 = parse_mill_levy_pdf_generic('../data/raw/kdor/20-22tableivavglevies.pdf', [2020, 2021, 2022]) 
print(len(df_2022_2024), len(df_2021_2023), len(df_2020_2022)) 

105 105 105


## Combine All Years Into One Clean Table 

In [33]:
def melt_mill_levy(df, years):
    records = []
    for _, row in df.iterrows():
        for year in years:
            records.append({
                'county': row['county'],
                'year': year,
                'rural_levy': float(row[f'rural_{year}']),
                'urban_levy': float(row[f'urban_{year}']),
                'county_avg_levy': float(row[f'county_avg_{year}'])
            })
    return pd.DataFrame(records)



In [34]:
long_2023_2025 = melt_mill_levy(df_2023_2025, [2023, 2024, 2025])
long_2022_2024 = melt_mill_levy(df_2022_2024, [2022, 2023, 2024])
long_2021_2023 = melt_mill_levy(df_2021_2023, [2021, 2022, 2023])
long_2020_2022 = melt_mill_levy(df_2020_2022, [2020, 2021, 2022])

In [35]:
all_levies = pd.concat([long_2023_2025, long_2022_2024, long_2021_2023, long_2020_2022], ignore_index=True)
all_levies = all_levies.drop_duplicates(subset=['county', 'year']).reset_index(drop=True)
len(all_levies)

630

## Runing just Reno County...Home County 

In [36]:
all_levies[all_levies['county'] == 'Reno']

,county,year,rural_levy,urban_levy,county_avg_levy
231,Reno,2023,142.234,160.386,153.117
232,Reno,2024,142.604,158.005,151.971
233,Reno,2025,144.229,161.486,154.778
392,Reno,2022,141.735,162.593,154.168
497,Reno,2021,144.019,168.638,158.504
602,Reno,2020,146.009,170.484,160.807


## Organizing year by descending order for clarity.

In [37]:
all_levies[all_levies['county'] == 'Reno'].sort_values('year') 

,county,year,rural_levy,urban_levy,county_avg_levy
602,Reno,2020,146.009,170.484,160.807
497,Reno,2021,144.019,168.638,158.504
392,Reno,2022,141.735,162.593,154.168
231,Reno,2023,142.234,160.386,153.117
232,Reno,2024,142.604,158.005,151.971
233,Reno,2025,144.229,161.486,154.778


## Parse Historical Statewide Valuation Data Testing whether rising assessed valuations, not levy rates, explain the tax increase.

In [41]:
with pdfplumber.open('../data/raw/kdor/PVDHistoricAssess.pdf') as pdf: 
    print(f'Pages: {len(pdf.pages)}') 
    print(pdf.pages[0].extract_text()[:1000])

Pages: 1
Assessed Value
Major Classes of Property (Billions)
Year Residential % of Total C&I Real/PP % of Total Utilities % of Total Ag Land % of Total Oil & Gas % of Total All Other % of Total Total Value
98 $7.365 39.00 $5.227 27.68 $2.870 15.20 $1.329 7.04 $1.455 7.70 $0.604 3.20 $18.849
99 $7.974 40.59 $5.713 29.08 $2.961 15.07 $1.351 6.88 $0.986 5.02 $0.622 3.17 $19.608
00 $8.766 41.91 $6.128 29.30 $2.919 13.95 $1.433 6.85 $0.937 4.48 $0.692 3.32 $20.875
01 $9.487 42.16 $6.402 28.45 $2.917 12.96 $1.553 6.90 $1.362 6.05 $0.737 3.28 $22.459
02 $10.092 43.72 $6.574 28.49 $2.817 12.20 $1.607 6.96 $1.201 5.20 $0.744 3.23 $23.035
03 $10.821 45.08 $6.847 28.53 $2.897 12.07 $1.563 6.51 $1.067 4.45 $0.764 3.19 $23.960
04 $11.467 45.06 $7.044 27.68 $3.055 12.00 $1.607 6.31 $1.457 5.72 $0.770 3.03 $25.398
05 $12.207 45.18 $7.405 27.41 $3.117 11.54 $1.593 5.90 $1.888 6.98 $0.809 2.99 $27.019
06 $13.083 45.09 $7.926 27.31 $3.105 10.70 $1.539 5.30 $2.456 8.46 $0.856 2.95 $28.964
07 $13.957 46.3

In [43]:
with pdfplumber.open('../data/raw/kdor/PVDHistoricAssess.pdf') as pdf: 
    full_text = pdf.pages[0].extract_text() 
    print(full_text[-1500:])

60 7.20 $0.879 2.94 $29.964
12 $14.609 48.02 $7.854 25.82 $3.557 11.69 $1.284 4.22 $2.204 7.25 $0.875 3.00 $30.383
13 $14.779 47.91 $7.958 25.80 $3.689 11.96 $1.447 4.69 $2.093 6.78 $0.883 2.86 $30.850
14 $15.279 47.25 $8.204 25.37 $3.737 11.56 $1.700 5.26 $2.112 6.53 $0.751 2.32 $31.783
15 $15.845 49.00 $8.488 26.25 $4.271 13.21 $1.974 6.10 $0.992 3.07 $0.765 2.37 $32.336
16 $16.495 53.47 $8.979 29.11 $4.219 13.68 $2.259 7.32 $0.463 1.50 $0.766 2.48 $33.181
17 $17.399 45.48 $9.225 24.11 $4.182 10.93 $2.555 6.68 $0.651 1.70 $0.771 2.02 $34.784
18 $18.364 50.18 $9.533 26.05 $4.439 12.13 $2.784 7.61 $0.698 1.91 $0.779 2.13 $36.597
19 $19.357 50.52 $9.788 25.69 $4.720 12.31 $2.908 7.59 $0.705 1.84 $0.777 2.05 $38.255
20 $20.361 51.76 $9.996 25.41 $4.883 12.41 $2.961 7.53 $0.322 0.82 $0.811 2.06 $39.334
21 $21.451 52.56 $10.028 24.57 $5.154 12.63 $2.949 7.23 $0.410 1.00 $0.820 2.01 $40.812
22 $24.052 53.97 $10.519 23.60 $5.306 11.91 $2.907 6.52 $0.848 1.90 $0.933 2.09 $44.565
23 $27.196 56

In [45]:
def parse_year_row(line):
    parts = line.split()
    if len(parts) < 2: 
        return None 
    year_token = parts[0]
    if not year_token.isdigit() or len(year_token) != 2:
        return None 
    if not parts[-1].startswith('$'):
        return None
    return parts

In [46]:
with pdfplumber.open('../data/raw/kdor/PVDHistoricAssess.pdf') as pdf:
    all_text = pdf.pages[0].extract_text()

raw_lines = all_text.split(chr(10))
valid_rows = []
for line in raw_lines:
    result = parse_year_row(line)
    if result is not None:
        valid_rows.append(result)

len(valid_rows)

28

### Convert to Clean DataFrame

In [47]:
def clean_year(token):
    year_num = int(token)
    if year_num >= 90:
        return 1900 + year_num
    else:
        return 2000 + year_num

In [48]:
assessed_records = []
for row in valid_rows:
    year = clean_year(row[0])
    total_value = float(row[-1].replace('$', '').replace(',', ''))
    assessed_records.append({'year': year, 'total_assessed_value_billions': total_value})

assessed_df = pd.DataFrame(assessed_records)
assessed_df

,year,total_assessed_value_billions
0,1998,18.849
1,1999,19.608
2,2000,20.875
3,2001,22.459
4,2002,23.035
5,2003,23.960
6,2004,25.398
7,2005,27.019
8,2006,28.964
9,2007,30.087


# Loaded into SQL

In [51]:
engine = create_engine('sqlite:///../data/processed/kansas_tax.db') 
all_levies.to_sql('mill_levies', engine, if_exists='replace', index=False) 
assessed_df.to_sql('statewide_valuation', engine, if_exists='replace', index=False)

28

In [52]:
pd.read_sql('SELECT * FROM mill_levies WHERE county = "Reno" ORDER BY year', engine)

,county,year,rural_levy,urban_levy,county_avg_levy
0,Reno,2020,146.009,170.484,160.807
1,Reno,2021,144.019,168.638,158.504
2,Reno,2022,141.735,162.593,154.168
3,Reno,2023,142.234,160.386,153.117
4,Reno,2024,142.604,158.005,151.971
5,Reno,2025,144.229,161.486,154.778


## Export data to csv due to powerbi not having sqllite capability

In [53]:
all_levies.to_csv('../data/processed/mill_levies.csv', index=False) 
assessed_df.to_csv('../data/processed/statewide_valuation.csv', index=False)

## Reno County Residential Valuation Growth (vs. Statewide)

Source: Kansas Open Gov, "Assessed Valuation Change on Existing Residential
Property" (originally from KDOR open records requests). Year-over-year %
change in assessed valuation on *existing* homes only — excludes new
construction, so this isolates real valuation growth rather than growth from
new building. Includes a State Avg row, enabling a direct Reno-vs-statewide
comparison. 2013–2023.

This is the piece that was missing from the flagship chart — a true
apples-to-apples Reno-specific valuation series to pair against the Reno
mill levy trend.

In [2]:
import pandas as pd
import sqlite3

valuation_change_raw = pd.read_csv("../data/raw/kdor/Assessed-Valuation-Change-on-Existing-Residential-Property.csv")
print(valuation_change_raw.shape)
valuation_change_raw.head()

(106, 12)


,County,2023,2022,2021,2020,2019,2018,2017,2016,2015,2014,2013
0,Allen,12.5,10.9,2.3,0.5,1.4,0.6,1.0,1.0,-0.4,-0.6,0.5
1,Anderson,10.8,27.8,0.0,1.7,2.4,2.5,-0.8,0.9,-0.1,0.2,-0.5
2,Atchison,13.2,12.1,3.3,1.6,1.2,0.0,0.2,-0.1,-0.2,-0.3,-0.2
3,Barber,4.7,2.1,0.5,-0.6,2.1,2.0,-0.2,1.2,3.4,2.8,2.6
4,Barton,18.0,6.8,1.8,-0.6,0.8,1.2,1.2,2.3,3.6,4.3,2.7


In [3]:
# Keep only Reno and the State Avg row for this comparison
reno_vs_state = valuation_change_raw[
    valuation_change_raw["County"].isin(["Reno", "State Avg"])
].copy()

# Melt from wide (one column per year) to long (one row per year)
reno_vs_state_long = reno_vs_state.melt(
    id_vars="County",
    var_name="year",
    value_name="valuation_change_pct"
)

reno_vs_state_long["year"] = reno_vs_state_long["year"].astype(int)
reno_vs_state_long = reno_vs_state_long.sort_values(["County", "year"]).reset_index(drop=True)

print(f"Rows: {len(reno_vs_state_long)}")
reno_vs_state_long

Rows: 22


,County,year,valuation_change_pct
0,Reno,2013,2.2
1,Reno,2014,2.4
2,Reno,2015,1.5
3,Reno,2016,1.6
4,Reno,2017,1.5
5,Reno,2018,1.6
6,Reno,2019,1.5
7,Reno,2020,2.5
8,Reno,2021,1.3
9,Reno,2022,8.9


## Validation

22 rows (Reno + State Avg, 2013–2023). Checked 2023 values against source:
Reno 9.7%, State Avg 12.0%.

In [5]:
conn = sqlite3.connect("../data/processed/kansas_tax.db")

reno_vs_state_long.to_sql("reno_valuation_vs_state", conn, if_exists="replace", index=False)
reno_vs_state_long.to_csv("../data/processed/reno_valuation_vs_state.csv", index=False)

check = pd.read_sql("SELECT * FROM reno_valuation_vs_state ORDER BY year DESC, County", conn)
print(check.head(6))

      County  year  valuation_change_pct
0       Reno  2023                   9.7
1  State Avg  2023                  12.0
2       Reno  2022                   8.9
3  State Avg  2022                  10.9
4       Reno  2021                   1.3
5  State Avg  2021                   4.4
